In [0]:
# Step 1 — create the Unity Catalog objects if they do not exist yet.
# SQL DDL is used (instead of bundle catalog resources) because this
# account uses Default Storage and API catalog creation requires an
# explicit managed location, while SQL CREATE CATALOG works.
for stmt in [
    "CREATE CATALOG IF NOT EXISTS retail_demo COMMENT 'Retail demo workspace for medallion architecture and DAB dev-to-prod training'",
    "CREATE SCHEMA IF NOT EXISTS retail_demo.bronze COMMENT 'Raw ingested data (append-only, audit columns, schema evolution)'",
    "CREATE SCHEMA IF NOT EXISTS retail_demo.silver COMMENT 'Cleansed, deduplicated, SCD1/SCD2 conformed tables'",
    "CREATE SCHEMA IF NOT EXISTS retail_demo.gold COMMENT 'Business-level aggregations as materialized views'",
    "CREATE SCHEMA IF NOT EXISTS retail_demo.raw COMMENT 'Landing zone volumes for source files'",
    "CREATE VOLUME IF NOT EXISTS retail_demo.raw.landing COMMENT 'Landing volume for CSV source files'",
]:
    spark.sql(stmt)
    print("OK:", stmt.split(" ")[1], stmt.split(" ")[3])

In [0]:
# Seeds the retail demo landing volume with the source CSV batches.
#
# Run ONCE per environment (the bundle defines a job for this notebook):
#   * two daily batches per entity where applicable
#   * C003 segment change -> SCD Type 2 history demo
#   * O1001 / OI26 duplicate re-drops -> dedup demo
#   * delivery_notes column in the 2nd orders batch -> schema evolution demo
#   * OI27 (qty 0), OI28 (qty -2), PM17 (negative), PM18 (crypto) -> DQ demos
import os

base = "/Volumes/retail_demo/raw/landing"

files = {}

files["stores/load_20261116.csv"] = """store_id,store_name,city,state,region,store_type
S01,Midtown Mart,New York,NY,Northeast,superstore
S02,Lakeside Bodega,Chicago,IL,Midwest,convenience
S03,Sunset Plaza,Los Angeles,CA,West,superstore
S04,Capitol Corner,Austin,TX,South,convenience
S05,Harbor Point,Seattle,WA,West,superstore
"""

files["products/load_20261116.csv"] = """product_id,product_name,category,brand,unit_price
P101,Espresso Beans 1kg,Grocery,Bean Bros,18.99
P102,Wireless Mouse,Electronics,LogiPro,24.50
P103,Cotton T-Shirt,Apparel,BasicThreads,9.99
P104,Yoga Mat,Sports,FlexFit,29.00
P105,LED Desk Lamp,Home,Lumina,15.75
P106,Stainless Thermos,Outdoor,TrailCo,22.40
P107,Notebook A5,Stationery,PaperTrail,4.25
P108,Bluetooth Speaker,Electronics,SonicWave,45.00
P109,Running Shoes,Sports,StrideMax,79.90
P110,Olive Oil 500ml,Grocery,Verde,11.30
"""

files["customers/load_20261116.csv"] = """customer_id,first_name,last_name,email,city,state,customer_segment,signup_date
C001,Alice,Johnson,alice.johnson@example.com,New York,NY,retail,2024-01-15
C002,Bob,Smith,bob.smith@example.com,Chicago,IL,retail,2024-02-20
C003,Carol,Williams,carol.williams@example.com,Austin,TX,retail,2024-03-05
C004,David,Brown,david.brown@example.com,Seattle,WA,premium,2024-04-10
C005,Eva,Davis,eva.davis@example.com,Los Angeles,CA,retail,2024-05-22
C006,Frank,Miller,frank.miller@example.com,Chicago,IL,retail,2024-06-30
C007,Grace,Lee,grace.lee@example.com,New York,NY,premium,2024-07-18
C008,Henry,Moore,henry.moore@example.com,Austin,TX,retail,2024-08-25
C009,Ivy,Taylor,ivy.taylor@example.com,Seattle,WA,retail,2024-09-12
C010,Jack,Anderson,jack.anderson@example.com,Los Angeles,CA,retail,2024-10-03
"""
files["customers/load_20261117.csv"] = """customer_id,first_name,last_name,email,city,state,customer_segment,signup_date
C001,Alice,Johnson,alice.johnson@example.com,New York,NY,retail,2024-01-15
C003,Carol,Williams,carol.williams@example.com,Austin,TX,premium,2024-03-05
C011,Karen,White,karen.white@example.com,Denver,CO,retail,2026-11-01
C012,Liam,Harris,liam.harris@example.com,Seattle,WA,premium,2026-11-10
"""

files["orders/load_20261116.csv"] = """order_id,customer_id,store_id,order_date,order_status,order_channel
O1001,C001,S01,2026-11-14,delivered,store
O1002,C002,S02,2026-11-14,delivered,online
O1003,C003,S04,2026-11-14,shipped,online
O1004,C004,S05,2026-11-15,delivered,store
O1005,C005,S03,2026-11-15,cancelled,online
O1006,C006,S02,2026-11-15,delivered,store
O1007,C007,S01,2026-11-15,delivered,online
O1008,C008,S04,2026-11-16,shipped,store
O1009,C009,S05,2026-11-16,delivered,online
O1010,C010,S03,2026-11-16,delivered,store
O1011,C001,S01,2026-11-16,shipped,store
O1012,C004,S05,2026-11-16,delivered,online
"""
files["orders/load_20261117.csv"] = """order_id,customer_id,store_id,order_date,order_status,order_channel,delivery_notes
O1001,C001,S01,2026-11-14,delivered,store,duplicate re-drop of load 1
O1013,C002,S02,2026-11-17,delivered,online,
O1014,C003,S04,2026-11-17,shipped,online,gift wrap requested
O1015,C007,S01,2026-11-17,delivered,store,
O1016,C011,S05,2026-11-17,shipped,online,leave at front desk
"""

files["order_items/load_20261116.csv"] = """order_item_id,order_id,product_id,quantity,unit_price,discount_pct
OI01,O1001,P101,2,18.99,0.00
OI02,O1001,P107,1,4.25,0.00
OI03,O1002,P102,1,24.50,5.00
OI04,O1003,P108,1,45.00,0.00
OI05,O1004,P104,2,29.00,10.00
OI06,O1005,P103,3,9.99,0.00
OI07,O1006,P106,1,22.40,0.00
OI08,O1007,P109,1,79.90,15.00
OI09,O1008,P110,2,11.30,0.00
OI10,O1009,P102,1,24.50,5.00
OI11,O1010,P101,1,18.99,0.00
OI12,O1011,P105,2,15.75,0.00
OI13,O1012,P104,1,29.00,0.00
OI14,O1004,P110,1,11.30,0.00
OI15,O1007,P107,2,4.25,0.00
OI16,O1003,P103,1,9.99,0.00
OI17,O1006,P102,1,24.50,0.00
OI18,O1009,P101,2,18.99,5.00
OI19,O1010,P107,3,4.25,0.00
OI20,O1012,P108,1,45.00,10.00
"""
files["order_items/load_20261117.csv"] = """order_item_id,order_id,product_id,quantity,unit_price,discount_pct
OI21,O1013,P110,2,11.30,0.00
OI22,O1013,P101,1,18.99,0.00
OI23,O1014,P102,1,24.50,5.00
OI24,O1015,P109,1,79.90,0.00
OI25,O1016,P101,1,18.99,0.00
OI26,O1013,P110,2,11.30,0.00
OI27,O1014,P104,0,29.00,0.00
OI28,O1015,P106,-2,22.40,0.00
"""

files["payments/load_20261116.csv"] = """payment_id,order_id,payment_method,payment_status,payment_amount,payment_ts
PM01,O1001,card,completed,42.23,2026-11-14T10:05:00
PM02,O1002,card,completed,23.28,2026-11-14T11:42:00
PM03,O1003,upi,completed,45.00,2026-11-14T13:10:00
PM04,O1004,card,completed,72.60,2026-11-15T09:20:00
PM05,O1005,card,refunded,29.97,2026-11-15T12:00:00
PM06,O1006,cash,completed,22.40,2026-11-15T15:30:00
PM07,O1007,card,completed,78.40,2026-11-15T17:45:00
PM08,O1008,upi,completed,22.60,2026-11-16T08:55:00
PM09,O1009,card,completed,23.28,2026-11-16T10:15:00
PM10,O1010,cash,completed,32.48,2026-11-16T12:40:00
PM11,O1011,card,pending,31.50,2026-11-16T14:05:00
PM12,O1012,card,completed,55.50,2026-11-16T16:30:00
"""
files["payments/load_20261117.csv"] = """payment_id,order_id,payment_method,payment_status,payment_amount,payment_ts
PM13,O1013,card,completed,52.58,2026-11-17T09:10:00
PM14,O1014,upi,completed,23.28,2026-11-17T10:35:00
PM15,O1015,card,completed,79.90,2026-11-17T11:55:00
PM16,O1016,upi,pending,18.99,2026-11-17T13:20:00
PM17,O1013,card,completed,-52.58,2026-11-17T14:00:00
PM18,O1014,crypto,completed,23.28,2026-11-17T14:30:00
"""

for rel, content in files.items():
    path = f"{base}/{rel}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

print("Landing volume seeded under", base)
for sub in sorted(os.listdir(base)):
    d = f"{base}/{sub}"
    print(" ", sub, "->", sorted(os.listdir(d)))